# Generate Evaluation Traces

This notebook creates realistic LLM traces across multiple prompt versions for a **Research Assistant** use case.

We'll:
1. Register 3 versions of the same prompt with meaningful differences
2. Run the same questions through each version
3. Compare outputs to see how prompt changes affect responses

**Use Case**: A research assistant that explains scientific concepts to different audiences.

In [ ]:
import os
import sys
from pathlib import Path

# Setup paths
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Database path
DB_PATH = str(project_root / "eval_traces.db")
os.environ["LLM_EVAL_DB_PATH"] = DB_PATH

print(f"Project root: {project_root}")
print(f"Database: {DB_PATH}")

In [ ]:
from llm_eval import LLMTracer, PromptRegistry
from llm_eval.store import TraceStore

# Initialize
registry = PromptRegistry(DB_PATH)
store = TraceStore(DB_PATH)

print("Registry and store initialized!")

## Define Prompt Versions

We'll create 3 versions of a "research_assistant" prompt with meaningful differences:

- **v1**: Basic - just answer the question
- **v2**: Structured - use headers, be concise, cite uncertainty
- **v3**: Audience-aware - adapt explanation to specified audience level

In [ ]:
# Version 1: Basic prompt
PROMPT_V1 = """You are a research assistant. Answer questions about scientific topics accurately and helpfully."""

# Version 2: Structured with uncertainty acknowledgment
PROMPT_V2 = """You are a research assistant specializing in scientific explanations.

Guidelines:
- Structure your response with clear sections when appropriate
- Be concise - aim for 2-3 paragraphs unless more detail is needed
- If uncertain about something, explicitly say so
- Distinguish between established facts and current research/hypotheses
- Use analogies to make complex concepts accessible"""

# Version 3: Audience-aware with formatting
PROMPT_V3 = """You are an expert research assistant who adapts explanations to your audience.

Guidelines:
- Assess the complexity of the question to gauge the asker's level
- For technical questions: be precise, use proper terminology, cite mechanisms
- For general questions: use analogies, avoid jargon, focus on intuition
- Structure with **bold headers** for multi-part answers
- Keep responses focused - 150-250 words unless the question requires more
- End with a brief "Key takeaway" sentence when helpful
- If the question is ambiguous, briefly address the most likely interpretation"""

# Register all versions
registry.register("research_assistant", PROMPT_V1, "Basic prompt - just answer questions")
registry.register("research_assistant", PROMPT_V2, "Structured with uncertainty acknowledgment")
registry.register("research_assistant", PROMPT_V3, "Audience-aware with formatting guidelines")

# Verify
versions = store.list_prompt_versions("research_assistant")
print(f"Registered {len(versions)} versions:")
for v in versions:
    print(f"  v{v['version']}: {v['description']}")

## Define Test Questions

Mix of question types to show how different prompts handle them:

In [ ]:
QUESTIONS = [
    # Basic factual
    "What causes the northern lights?",
    
    # Requires nuance/uncertainty
    "Why do we dream?",
    
    # Technical concept
    "How does CRISPR gene editing work?",
    
    # Comparison question
    "What's the difference between machine learning and deep learning?",
    
    # Current research area
    "What do we know about how exercise affects brain health?",
    
    # Multi-part question
    "What is quantum entanglement and why is it important for quantum computing?",
]

print(f"Prepared {len(QUESTIONS)} test questions")

## Setup LLM

Configure the LLM client. Update the model/API as needed for your environment.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

# Configure your LLM - adjust model name as needed
# For OpenAI:
# llm_base = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

# For Azure OpenAI or other providers, configure accordingly
llm_base = ChatOpenAI(
    model="gpt-4o-mini",  # or your model
    temperature=0.7,
)

print(f"LLM configured: {llm_base.model_name}")

## Generate Traces

Run each question through each prompt version and trace the results.

In [ ]:
PROJECT = "research_assistant_eval"

def run_traced_query(tracer, system_prompt: str, question: str):
    """Run a query with tracing enabled."""
    llm = ChatOpenAI(
        model="gpt-4o-mini",
        temperature=0.7,
        callbacks=[tracer]
    )
    
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=question)
    ]
    
    response = llm.invoke(messages)
    return response.content

print("Query function ready")

In [ ]:
# Run all questions through Version 1
print("=" * 60)
print("RUNNING VERSION 1: Basic prompt")
print("=" * 60)

tracer_v1 = LLMTracer(db_path=DB_PATH, project=PROJECT, session_id="v1_basic")
tracer_v1.set_prompt_context("research_assistant", version=1)

for i, question in enumerate(QUESTIONS):
    print(f"\n[Q{i+1}] {question}")
    response = run_traced_query(tracer_v1, PROMPT_V1, question)
    print(f"[A] {response[:200]}..." if len(response) > 200 else f"[A] {response}")

print("\n✓ Version 1 complete")

In [ ]:
# Run all questions through Version 2
print("=" * 60)
print("RUNNING VERSION 2: Structured with uncertainty")
print("=" * 60)

tracer_v2 = LLMTracer(db_path=DB_PATH, project=PROJECT, session_id="v2_structured")
tracer_v2.set_prompt_context("research_assistant", version=2)

for i, question in enumerate(QUESTIONS):
    print(f"\n[Q{i+1}] {question}")
    response = run_traced_query(tracer_v2, PROMPT_V2, question)
    print(f"[A] {response[:200]}..." if len(response) > 200 else f"[A] {response}")

print("\n✓ Version 2 complete")

In [ ]:
# Run all questions through Version 3
print("=" * 60)
print("RUNNING VERSION 3: Audience-aware")
print("=" * 60)

tracer_v3 = LLMTracer(db_path=DB_PATH, project=PROJECT, session_id="v3_audience_aware")
tracer_v3.set_prompt_context("research_assistant", version=3)

for i, question in enumerate(QUESTIONS):
    print(f"\n[Q{i+1}] {question}")
    response = run_traced_query(tracer_v3, PROMPT_V3, question)
    print(f"[A] {response[:200]}..." if len(response) > 200 else f"[A] {response}")

print("\n✓ Version 3 complete")

## Summary Statistics

In [ ]:
# Check what we created
sessions = store.get_sessions(project=PROJECT)

print("Sessions created:")
print("-" * 50)
for s in sessions:
    print(f"  {s['session_id']}: {s['trace_count']} traces, {s['total_tokens']} tokens")

total_traces = sum(s['trace_count'] for s in sessions)
total_tokens = sum(s['total_tokens'] or 0 for s in sessions)

print(f"\nTotal: {total_traces} traces, {total_tokens:,} tokens")

## Compare Outputs Side-by-Side

Let's compare how the same question was answered differently across versions:

In [ ]:
def compare_versions_for_question(question_index: int):
    """Show how different prompt versions answered the same question."""
    question = QUESTIONS[question_index]
    print(f"Question: {question}")
    print("=" * 70)
    
    for session_id, version in [("v1_basic", 1), ("v2_structured", 2), ("v3_audience_aware", 3)]:
        traces = store.get_traces({"session_id": session_id})
        traces.sort(key=lambda x: x['created_at'])
        
        if question_index < len(traces):
            trace = traces[question_index]
            output = trace.get('output_content', '')[:500]
            tokens = trace.get('total_tokens', 0)
            
            print(f"\n--- VERSION {version} ({tokens} tokens) ---")
            print(output)
            if len(trace.get('output_content', '')) > 500:
                print("...")

# Compare responses to "Why do we dream?" (shows uncertainty handling)
compare_versions_for_question(1)

In [ ]:
# Compare responses to "How does CRISPR work?" (technical question)
compare_versions_for_question(2)

In [ ]:
# Compare responses to the multi-part quantum entanglement question
compare_versions_for_question(5)

## Launch Dashboard

Now you can view and annotate these traces in the dashboard!

In [ ]:
print("To launch the dashboard:")
print()
print(f"  LLM_EVAL_DB_PATH={DB_PATH} llm-eval-dashboard")
print()
print("Or for SageMaker:")
print(f"  SAGEMAKER_PROXY=1 LLM_EVAL_DB_PATH={DB_PATH} llm-eval-dashboard")

## Add Some Annotations (Optional)

Pre-populate some annotations to test the Failure Analysis page:

In [ ]:
from llm_eval.models import Annotation
import random

# Get all traces
all_traces = store.get_traces({"project": PROJECT})

# Annotate some traces
annotations_added = 0
for trace in all_traces:
    # Skip if already annotated
    if store.get_annotation(trace['id']):
        continue
    
    # Annotate ~60% of traces
    if random.random() > 0.6:
        continue
    
    # Simulate realistic ratings based on version
    version = trace['prompt_version']
    
    # V1 (basic) has lower quality
    if version == 1:
        rating = "good" if random.random() < 0.5 else "bad"
        notes = random.choice([
            "Too brief, lacks context",
            "Adequate but could be more structured",
            "Missing important nuances",
            "Good basic answer",
        ]) if rating == "bad" else "Acceptable response"
    # V2 (structured) is better
    elif version == 2:
        rating = "good" if random.random() < 0.75 else "bad"
        notes = random.choice([
            "Good structure and clarity",
            "Appropriately acknowledges uncertainty",
            "Well organized",
        ]) if rating == "good" else "Could use better analogies"
    # V3 (audience-aware) is best
    else:
        rating = "good" if random.random() < 0.85 else "bad"
        notes = random.choice([
            "Excellent adaptation to audience",
            "Clear key takeaway",
            "Perfect level of detail",
            "Great use of formatting",
        ]) if rating == "good" else "Slightly verbose"
    
    store.save_annotation(Annotation(
        trace_id=trace['id'],
        rating=rating,
        notes=notes,
        annotator="auto"
    ).model_dump())
    annotations_added += 1

print(f"Added {annotations_added} annotations")

In [ ]:
# Show annotation statistics per version
print("Annotation Statistics by Version:")
print("-" * 50)

for version in [1, 2, 3]:
    traces = [t for t in all_traces if t['prompt_version'] == version]
    good = sum(1 for t in traces if store.get_annotation(t['id']) and store.get_annotation(t['id'])['rating'] == 'good')
    bad = sum(1 for t in traces if store.get_annotation(t['id']) and store.get_annotation(t['id'])['rating'] == 'bad')
    total = good + bad
    
    if total > 0:
        print(f"  Version {version}: {good}/{total} good ({good/total*100:.0f}% pass rate)")
    else:
        print(f"  Version {version}: No annotations yet")

## Done!

You now have:
- 3 prompt versions with meaningful differences
- 6 questions × 3 versions = 18 traces
- Pre-populated annotations showing version performance

Launch the dashboard to:
1. **Sessions page**: View conversations, add your own annotations
2. **Versions page**: Compare v1 vs v2 vs v3 pass rates
3. **Failures page**: Analyze what went wrong with bad responses